# **UGST4058 — Introduction to Public Policy 2025/26 Winter**
# Central European University
## API Request on Air Quality Data

Quantitative Methods Seminar
- Liza Drini
- Benedek Szalma
- Zeteny Cseresznyes  

### 0. Key

In [ ]:
api_key = "36247A98-124B-11F1-B596-4201AC1DC123"

### 1. Modules

In [ ]:
import requests
import time
import calendar
import pandas as pd
from datetime import datetime, timezone
from io import StringIO
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import Point

### 2. Setup

In [ ]:
api_key = api_key
lat_nw, lng_nw = 45.55, 8.95
lat_se, lng_se = 45.35, 9.40
fields = "pm2.5_atm,pm2.5_atm_a,pm2.5_atm_b,pm10.0_atm,humidity,temperature,pressure"
history_url_template = "https://api.purpleair.com/v1/sensors/{sensor_index}/history/csv"
headers = {"X-API-Key": api_key}

### 3. API Request Pipeline

In [ ]:
def fetch_with_retry(url, headers, params, retries=3, wait=5):
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=30)
            return resp
        except Exception as e:
            print(f"  ⚠ Attempt {attempt+1} failed: {e}")
            time.sleep(wait * (attempt + 1))  # progressive backoff: 5s, 10s, 15s
    return None

#### 3.1. Find Sensors and Map

In [ ]:
sensor_params = {
    "fields": "sensor_index,name,latitude,longitude,location_type",
    "nwlat": lat_nw,
    "nwlng": lng_nw,
    "selat": lat_se,
    "selng": lng_se,
}

response = fetch_with_retry("https://api.purpleair.com/v1/sensors", headers=headers, params=sensor_params)

if response and response.status_code == 200:
    sensors_data = response.json()
    sensor_list = sensors_data.get("data", [])
    fields_meta = sensors_data["fields"]
    idx_col = fields_meta.index("sensor_index")
    name_col = fields_meta.index("name")
    lat_col = fields_meta.index("latitude")
    lng_col = fields_meta.index("longitude")

    sensor_indices = [s[idx_col] for s in sensor_list]

    print(f"✓ Found {len(sensor_indices)} sensors in Milan:\n")
    for s in sensor_list:
        print(f"  ID: {s[idx_col]} | Name: {s[name_col]} | Lat: {s[lat_col]}, Lng: {s[lng_col]}")
else:
    print("✗ Failed to fetch sensors")
    sensor_indices = []

sensors_df = pd.DataFrame(sensor_list, columns=fields_meta)

# map
m = folium.Map(location=[45.4642, 9.1900], zoom_start=11, tiles="CartoDB positron")

for _, row in sensors_df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=8,
        color="#e63946",
        fill=True,
        fill_color="#e63946",
        fill_opacity=0.8,
        tooltip=f"<b>{row['name']}</b><br>ID: {row['sensor_index']}<br>Lat: {row['latitude']}<br>Lng: {row['longitude']}"
    ).add_to(m)

m.save("../Figures/milan_sensors_map.html")
m

#### 3.2. Generate Date Ranges

In [ ]:
# --- Step 2: Generate monthly date ranges ---
date_ranges = []
for year in range(2022, 2026):
    for month in range(1, 13):
        start = datetime(year, month, 1, tzinfo=timezone.utc)
        last_day = calendar.monthrange(year, month)[1]
        end = datetime(year, month, last_day, tzinfo=timezone.utc)
        date_ranges.append((start, end))

#### 3.3. Fetch Historical Data

In [ ]:
all_dataframes = {}

for sensor_index in sensor_indices:
    print(f"\n--- Starting sensor {sensor_index} ---")
    sensor_chunks = []

    for start, end in date_ranges:
        params = {
            "start_timestamp": int(start.timestamp()),
            "end_timestamp": int(end.timestamp()),
            "average": 1440,
            "fields": fields,
        }

        url = history_url_template.format(sensor_index=sensor_index)
        resp = fetch_with_retry(url, headers=headers, params=params)

        if resp is None:
            print(f"  ✗ {start.strftime('%B %Y')} — gave up after retries")
        elif resp.status_code == 200:
            df_chunk = pd.read_csv(StringIO(resp.text))
            sensor_chunks.append(df_chunk)
            print(f"  ✓ {start.strftime('%B %Y')} — {len(df_chunk)} records fetched")
        else:
            print(f"  ✗ {start.strftime('%B %Y')} — FAILED: {resp.status_code} - {resp.text}")

        time.sleep(2)  # rate limit

    if sensor_chunks:
        all_dataframes[sensor_index] = pd.concat(sensor_chunks).sort_values("time_stamp").reset_index(drop=True)
        print(f"  ✓ Sensor {sensor_index} complete — {len(all_dataframes[sensor_index])} total records")

#### 3.4. Save to CSV

In [ ]:
# --- Step 4: Combine and save ---
if all_dataframes:
    combined = pd.concat(
        [df.assign(sensor_index=idx) for idx, df in all_dataframes.items()],
        ignore_index=True
    )
    combined.to_csv("../Data/milan_air_quality.csv", index=False)
    print(f"\n✓ Saved {len(combined)} rows to milan_air_quality.csv")
else:
    print("\n✗ No data to save.")